# 02a — Correlation and Redundancy

## The One-Sentence Version
If two features move together, carrying both is carrying a partial duplicate.

## What You'll Build Intuition For
- Why correlated features are partially redundant
- How to read a correlation matrix and spot block structure
- How eigenvalues reveal the *effective* number of dimensions

## Prerequisites
- [01a — Building Intuition](../01_what_is_a_dimension/01a_building_intuition.ipynb)
- [01b — The Curse of Dimensionality](../01_what_is_a_dimension/01b_curse_of_dimensionality.ipynb)

## The Story

In Module 01 we learned that dimensions are just questions, and that high-dimensional space is hostile. But there's a saving grace hiding in plain sight: most of your dimensions aren't independent.

Think about height and weight. If you know someone is 6'2", you already have a strong guess about their weight — not a perfect guess, but you've narrowed it down considerably. The two measurements are *correlated*. And that means carrying both is partially redundant — like photocopying a document and filing both copies.

This redundancy isn't a flaw. It's a gift. It means your data doesn't really live in as many dimensions as your spreadsheet suggests. Find the redundancy, and you've found the first handle for dimensionality reduction.

This notebook teaches you to spot redundancy, measure it, and count how many *real* dimensions your data has.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS

apply_style()
rng = np.random.default_rng(42)

## Two Features, One Signal

Let's generate a dataset where height and weight are correlated at r ≈ 0.85. Both are measuring aspects of the same underlying thing — body size.

In [ ]:
# Generate height and weight with strong correlation
n = 200
body_size = rng.standard_normal(n)  # latent "body size" factor

height_cm = 170 + 10 * body_size + rng.standard_normal(n) * 3
weight_kg = 75 + 12 * body_size + rng.standard_normal(n) * 5

r = np.corrcoef(height_cm, weight_kg)[0, 1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(height_cm, weight_kg, s=30, alpha=0.6,
           color=COLOURS["signal"], edgecolors="white", linewidth=0.3)
ax.set_xlabel("Height (cm)")
ax.set_ylabel("Weight (kg)")
ax.set_title(f"Height vs Weight (r = {r:.2f})")

# Add trend line
z = np.polyfit(height_cm, weight_kg, 1)
x_line = np.linspace(height_cm.min(), height_cm.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), "--",
        color=COLOURS["accent"], linewidth=2, alpha=0.8)

plt.tight_layout()
plt.savefig("visuals/02a_height_weight.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Correlation: {r:.2f}")
print(f"If I know someone's height, I can predict ~{r**2:.0%} of the variance in their weight.")
print(f"That means weight only adds ~{1 - r**2:.0%} of *new* information beyond height.")

That's the core idea: **correlation ≠ 0 means partial redundancy.** The higher the correlation, the less new information the second feature adds. If r = 1.0, the second feature is a perfect copy — zero new information. If r = 0, the features are fully independent — each carries unique information.

Most real-world features fall somewhere in between. The question is: *how much redundancy is there across all your features?*

## The Correlation Matrix

With 2 features, you just look at one number. With 10 or 100 features, you need the full correlation matrix — a grid showing how every pair of features moves together.

Let's generate 10 features that have a hidden block structure: three groups of correlated features, plus one independent feature.

In [ ]:
# Generate 10 features with block correlation structure
# 3 latent factors → 3 clusters of correlated features + 1 independent

n = 300
factor_1 = rng.standard_normal(n)  # drives features 0, 1, 2
factor_2 = rng.standard_normal(n)  # drives features 3, 4, 5
factor_3 = rng.standard_normal(n)  # drives features 6, 7, 8

features = np.column_stack([
    # Cluster 1: features driven by factor_1
    factor_1 + rng.standard_normal(n) * 0.3,
    factor_1 * 0.8 + rng.standard_normal(n) * 0.4,
    factor_1 * 1.2 + rng.standard_normal(n) * 0.3,
    # Cluster 2: features driven by factor_2
    factor_2 + rng.standard_normal(n) * 0.3,
    factor_2 * 0.9 + rng.standard_normal(n) * 0.35,
    factor_2 * 1.1 + rng.standard_normal(n) * 0.3,
    # Cluster 3: features driven by factor_3
    factor_3 + rng.standard_normal(n) * 0.4,
    factor_3 * 0.7 + rng.standard_normal(n) * 0.5,
    factor_3 * 1.3 + rng.standard_normal(n) * 0.3,
    # Independent feature
    rng.standard_normal(n),
])

feature_names = [f"F{i}" for i in range(10)]

# Correlation heatmap
corr = np.corrcoef(features.T)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(feature_names)
ax.set_yticklabels(feature_names)
ax.set_title("Correlation Matrix: 10 features, 3 clusters + 1 independent")

# Annotate values
for i in range(10):
    for j in range(10):
        color = "white" if abs(corr[i, j]) > 0.5 else "black"
        ax.text(j, i, f"{corr[i, j]:.2f}", ha="center", va="center",
                fontsize=7, color=color)

# Draw block boundaries
for pos in [2.5, 5.5, 8.5]:
    ax.axhline(pos, color="black", linewidth=1.5)
    ax.axvline(pos, color="black", linewidth=1.5)

plt.colorbar(im, ax=ax, label="Correlation", shrink=0.8)
plt.tight_layout()
plt.savefig("visuals/02a_correlation_blocks.png", dpi=150, bbox_inches="tight")
plt.show()

See the block structure? Features 0-2 correlate strongly with each other (they're all driven by the same underlying factor), same for 3-5 and 6-8. Feature 9 is independent — low correlation with everything else.

**10 measurements, but only ~4 independent signals.** The three feature clusters are each saying the same thing in slightly different ways.

## Effective Dimensionality

The correlation matrix tells us *which* features are redundant. But how do we put a single number on it — how many *effective* dimensions does this data actually have?

The answer comes from eigenvalues. When we decompose the correlation matrix, each eigenvalue tells us how much variance one independent direction captures. If most of the variance is in a few eigenvalues, the effective dimensionality is low.

In [ ]:
# Eigenvalue decomposition of the correlation matrix
eigenvalues = np.linalg.eigvalsh(corr)[::-1]  # descending order
explained = eigenvalues / eigenvalues.sum()
cumulative = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Eigenvalue bar chart
colours = [COLOURS["signal"] if e > 1.0 else COLOURS["neutral"] for e in eigenvalues]
ax1.bar(range(1, 11), eigenvalues, color=colours, alpha=0.8, edgecolor="white")
ax1.axhline(y=1.0, color=COLOURS["accent"], linestyle="--", alpha=0.7,
            label="Kaiser criterion (λ=1)")
ax1.set_xlabel("Eigenvalue index")
ax1.set_ylabel("Eigenvalue")
ax1.set_title("Eigenvalues of Correlation Matrix")
ax1.legend()

# Cumulative explained variance
ax2.plot(range(1, 11), cumulative, "o-",
         color=COLOURS["signal"], markersize=8, linewidth=2)
ax2.axhline(y=0.90, color=COLOURS["accent"], linestyle="--", alpha=0.7,
            label="90% threshold")
ax2.set_xlabel("Number of components")
ax2.set_ylabel("Cumulative variance explained")
ax2.set_title("How many dimensions do we actually need?")
ax2.legend()
ax2.set_ylim(0, 1.05)

n_90 = np.searchsorted(cumulative, 0.90) + 1
ax2.axvline(n_90, color=COLOURS["accent"], linewidth=0.8, alpha=0.4)

plt.tight_layout()
plt.savefig("visuals/02a_eigenvalues.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Eigenvalues: {eigenvalues.round(2)}")
print(f"\n{n_90} components capture {cumulative[n_90-1]:.0%} of the variance")
print(f"10 measurements → ~{n_90} effective dimensions")

The eigenvalue plot tells the whole story: 3-4 eigenvalues are large (above the Kaiser criterion line of λ=1), and the rest are small. That maps perfectly to our construction: 3 latent factors + 1 independent feature.

**10 measurements, but really only ~4 independent signals.** The rest is redundancy.

## A Practical Rule of Thumb

Here's a useful heuristic: if |correlation| > 0.8 between two features, they're substantially redundant. You can count clusters of highly correlated features as single effective dimensions.

Let's do this manually, then verify with PCA.

In [ ]:
# Manual cluster counting: features with |r| > 0.8
threshold = 0.8
print("Feature pairs with |correlation| > 0.8:")
for i in range(10):
    for j in range(i + 1, 10):
        if abs(corr[i, j]) > threshold:
            print(f"  F{i} ↔ F{j}: r = {corr[i, j]:.2f}")

print(f"\nManual count: 3 clusters (F0-F2, F3-F5, F6-F8) + 1 independent (F9) = 4 effective dims")

# PCA verification
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(features)
pca = PCA(random_state=42).fit(X_scaled)

n_95 = np.searchsorted(np.cumsum(pca.explained_variance_ratio_), 0.95) + 1
print(f"\nPCA answer: {n_95} components for 95% of variance")
print("The manual rule of thumb and PCA agree: about 4 real dimensions.")

> **Key insight:** Redundancy isn't a bug — it's your best friend. Every pair of correlated features is a pair that's partially encoding the same thing. The more redundancy, the more room to compress. This is the engine that drives everything in Modules 03–06.

<details>
<summary><b>The Maths Behind This</b></summary>

The correlation matrix $\mathbf{R}$ of a $d$-dimensional dataset is a $d \times d$ symmetric positive semi-definite matrix. Its eigendecomposition:

$$\mathbf{R} = \mathbf{Q} \mathbf{\Lambda} \mathbf{Q}^T$$

produces eigenvalues $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_d \geq 0$ that sum to $d$ (since diagonal of a correlation matrix is all 1s).

If the data has $k$ independent signals, approximately $k$ eigenvalues will be large and the remaining $d - k$ will be near zero. The **effective dimensionality** can be estimated by:
- Counting eigenvalues above 1 (Kaiser criterion)
- Finding the "elbow" in the scree plot
- Finding the number of components for 90-95% cumulative variance

All three methods are heuristics, but they usually agree to within ±1.

</details>

## Where This Matters

**Clinical data:** Lab panels measure many things, but liver function tests (ALT, AST, ALP, bilirubin) are heavily correlated — they're all measuring "how is the liver doing?" A 50-feature patient record might have only 8-10 independent clinical signals.

**Sensor networks:** Dozens of sensors in a system often track the same physical process. Temperature, pressure, and flow rate in a pipe are correlated because they're all driven by the same underlying fluid dynamics.

## Feynman Check

1. **If features A, B, C all correlate at r > 0.9, how many real dimensions do they represent?**

2. **Sketch (in your head) a correlation matrix that shows 20 features but only 4 independent signals. What pattern would you see?**

Now that you can spot redundancy, let's formalise the distinction between the space data lives *in* and the space it moves *through* in **02b — Intrinsic vs Ambient Dimension**.